# Popular Times for Times Square Restaurants

This notebook fetches restaurant data near **Times Square, NYC** using:

1. **Google Places API (New)** — to find nearby restaurants and get details
2. **`populartimes` package** — to scrape popular times data from Google Maps

**Requirements:**
- A Google API key with **Places API (New)** enabled
- `populartimes` package: `pip install --use-pep517 git+https://github.com/m-wrzr/populartimes.git`

**Note:** The `populartimes` package scrapes Google Maps search results for popular times data. This works best in browser-like environments (Google Colab). From server environments, Google may not return popular times data.

In [ ]:
# Install populartimes (uncomment if needed)
# !pip install --use-pep517 git+https://github.com/m-wrzr/populartimes.git

In [ ]:
import json
import requests
import pandas as pd

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    API_KEY = input("Enter your Google API key: ")

## Step 1: Find restaurants near Times Square using Nearby Search

Uses the [Places API (New)](https://developers.google.com/maps/documentation/places/web-service/nearby-search) to search within 300m of Times Square center (40.758, -73.9855).

In [ ]:
url = 'https://places.googleapis.com/v1/places:searchNearby'
headers = {
    'X-Goog-Api-Key': API_KEY,
    'X-Goog-FieldMask': ','.join([
        'places.id', 'places.displayName', 'places.formattedAddress',
        'places.rating', 'places.userRatingCount', 'places.types',
        'places.location', 'places.primaryType',
        'places.regularOpeningHours', 'places.priceLevel',
    ]),
    'Content-Type': 'application/json'
}
body = {
    'includedTypes': ['restaurant'],
    'maxResultCount': 20,
    'locationRestriction': {
        'circle': {
            'center': {'latitude': 40.758, 'longitude': -73.9855},
            'radius': 300.0
        }
    }
}

resp = requests.post(url, headers=headers, json=body)
resp.raise_for_status()
places_data = resp.json()

places = places_data.get('places', [])
print(f"Found {len(places)} restaurants near Times Square")

In [ ]:
# Build a summary DataFrame
rows = []
for p in places:
    rows.append({
        'name': p.get('displayName', {}).get('text'),
        'place_id': p.get('id'),
        'address': p.get('formattedAddress'),
        'rating': p.get('rating'),
        'reviews': p.get('userRatingCount'),
        'price_level': p.get('priceLevel'),
        'primary_type': p.get('primaryType'),
        'lat': p.get('location', {}).get('latitude'),
        'lng': p.get('location', {}).get('longitude'),
        'open_now': p.get('regularOpeningHours', {}).get('openNow'),
    })

df_places = pd.DataFrame(rows).sort_values('reviews', ascending=False)
df_places

## Step 2: Get Popular Times via `populartimes`

The `populartimes` package uses two approaches:
- `get_id(api_key, place_id)` — gets details via Places API + scrapes Google for popular times
- `get(api_key, types, sw, ne)` — area search (uses deprecated Radar Search API, **no longer works**)

We use `get_id()` with the place IDs we found above.

In [ ]:
import populartimes

# Fetch popular times for each restaurant
pt_results = []
for _, row in df_places.head(10).iterrows():
    name = row['name']
    place_id = row['place_id']
    try:
        detail = populartimes.get_id(API_KEY, place_id)
        has_pt = detail.get('populartimes') is not None
        pt_results.append({
            'name': name,
            'place_id': place_id,
            'has_populartimes': has_pt,
            'current_popularity': detail.get('current_popularity'),
            'time_spent': detail.get('time_spent'),
            'detail': detail,
        })
        status = 'YES' if has_pt else 'no'
        print(f"  {name}: populartimes={status}, current={detail.get('current_popularity', 'N/A')}")
    except Exception as e:
        print(f"  {name}: ERROR - {e}")
        pt_results.append({'name': name, 'place_id': place_id, 'has_populartimes': False, 'detail': None})

print(f"\n{sum(1 for r in pt_results if r['has_populartimes'])} / {len(pt_results)} have popular times data")

In [ ]:
# Show detailed popular times for places that have them
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

for r in pt_results:
    if not r.get('has_populartimes') or r.get('detail') is None:
        continue

    detail = r['detail']
    print(f"\n=== {detail['name']} ===")
    print(f"Current popularity: {detail.get('current_popularity', 'N/A')}")
    print(f"Time spent: {detail.get('time_spent', 'N/A')} minutes")

    pt_rows = []
    for day_data in detail['populartimes']:
        for hour, popularity in enumerate(day_data['data']):
            pt_rows.append({'day': day_data['name'], 'hour': hour, 'popularity': popularity})

    df_pt = pd.DataFrame(pt_rows)
    pivot = df_pt.pivot(index='hour', columns='day', values='popularity')[days]
    display(pivot)

    # Plot
    pivot.plot(figsize=(14, 5), title=f"Popular Times — {detail['name']}")

## Step 3: Alternative — Direct Google Maps Scraping

If `populartimes.get_id()` fails (requires the legacy Places API), we can use the internal scraping function directly. This doesn't need an API key — it scrapes Google search results.

**Note:** This approach is fragile and may break when Google changes their page structure. It also works best from browser-like environments.

In [ ]:
from populartimes.crawler import get_populartimes_from_search, get_popularity_for_day

# Try direct scraping for each restaurant
scrape_results = []
for _, row in df_places.head(10).iterrows():
    name = row['name']
    address = row['address']
    try:
        rating, rating_n, popular_times, current_popularity, time_spent = get_populartimes_from_search(name, address)
        has_pt = popular_times is not None
        scrape_results.append({
            'name': name,
            'rating': rating,
            'rating_n': rating_n,
            'has_populartimes': has_pt,
            'current_popularity': current_popularity,
            'time_spent': time_spent,
            'popular_times_raw': popular_times,
        })
        print(f"  {name}: rating={rating}, populartimes={'YES' if has_pt else 'no'}, current={current_popularity}")
    except Exception as e:
        print(f"  {name}: ERROR - {e}")

print(f"\n{sum(1 for r in scrape_results if r.get('has_populartimes'))} / {len(scrape_results)} have popular times data")

In [ ]:
# Parse and display any scraped popular times
for r in scrape_results:
    if not r.get('has_populartimes'):
        continue

    pop, wait = get_popularity_for_day(r['popular_times_raw'])

    print(f"\n=== {r['name']} ===")
    print(f"Rating: {r['rating']}, Reviews: {r['rating_n']}")
    print(f"Time spent: {r['time_spent']} min")

    pt_rows = []
    for day_data in pop:
        for hour, popularity in enumerate(day_data['data']):
            pt_rows.append({'day': day_data['name'], 'hour': hour, 'popularity': popularity})

    df_pt = pd.DataFrame(pt_rows)
    pivot = df_pt.pivot(index='hour', columns='day', values='popularity')[days]
    display(pivot)
    pivot.plot(figsize=(14, 5), title=f"Popular Times — {r['name']}")

## Summary

| Method | API Key Required | Popular Times | Status |
|--------|-----------------|---------------|--------|
| `populartimes.get()` (area search) | Yes (legacy Places API) | Yes | **Broken** — uses deprecated Radar Search API |
| `populartimes.get_id()` (single place) | Yes (legacy Places API) | Yes | Works if legacy Places API is enabled |
| Direct scraping (`get_populartimes_from_search`) | No | Sometimes | Fragile — depends on Google's HTML structure |
| Places API (New) Nearby Search | Yes (Places API New) | No | **Works** — but doesn't include popular times |

### Recommendations
- For **finding restaurants**: Use the Places API (New) — reliable and well-documented
- For **popular times**: Use `populartimes.get_id()` with a key that has the legacy Places API enabled, or run the scraping from Google Colab (browser-like environment)
- Consider **[Outscraper](https://outscraper.com/)** or **[SerpApi](https://serpapi.com/)** as more reliable alternatives for popular times data